# 03d — 2SLS (hovedmodell)

To-stegs minste kvadraters metode med temperatur og nedbør som instrumenter for `cons_NO4`. HAC-standardfeil (Bartlett-kjerne, 24 lag).

**Input:** `intermediate/df_iso.parquet`

**Output:** `intermediate/preds_tsls.parquet`, `intermediate/models_tsls.pkl`

In [1]:
import pandas as pd

from src.config import (
    INTERMEDIATE_DIR, TARGET, IV_ENDOG, IV_INSTRUMENTS, IV_EXOG,
    TRAIN_YEARS, TEST_YEARS, apply_style,
)
from src.evaluation import eval_metrics
from src.model_training import (
    load_prepared_iso_data, get_split_masks, make_prediction_frame,
    save_model_artifacts, fit_2sls, predict_2sls,
)

apply_style()

In [2]:
df_iso = load_prepared_iso_data(INTERMEDIATE_DIR)
train_mask, test_mask = get_split_masks(df_iso, TRAIN_YEARS, TEST_YEARS)

print(f"Trening: {train_mask.sum():,} ISO-timer")
print(f"Test:    {test_mask.sum():,} ISO-timer")
print(f"Endogen: {IV_ENDOG}")
print(f"Instrumenter: {IV_INSTRUMENTS}")

Trening: 25,072 ISO-timer
Test:    9,844 ISO-timer
Endogen: cons_NO4
Instrumenter: ['temp_NO4', 'precip_NO4']


In [3]:
tsls_model, work_train = fit_2sls(
    df=df_iso.loc[train_mask],
    y_col=TARGET,
    endog_col=IV_ENDOG,
    instrument_cols=IV_INSTRUMENTS,
    exog_cols=IV_EXOG,
    hac_lags=24,
)

print(tsls_model.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:              price_NO4   R-squared:                      0.2229
Estimator:                    IV-2SLS   Adj. R-squared:                 0.2217
No. Observations:               24468   F-statistic:                    361.57
Date:                Tue, May 05 2026   P-value (F-stat)                0.0000
Time:                        11:56:40   Distribution:                 chi2(37)
Cov. Estimator:                kernel                                         
                                                                              
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
const            -1205.7     211.61    -5.6979     0.0000     -1620.5     -790.97
fill_avvik       -481.73     137.40    -

## Førstesteg- og overidentifiseringsdiagnostikk

Førstesteg-F skal være > 10 (Staiger-Stock) for å sikre at instrumentene ikke er svake. Sargan-testen (overidentifisering) sjekker om instrumentene som settes er konsistente med eksogenitet.

In [4]:
print("==== Førstesteg ====")
print(tsls_model.first_stage)
print("\n==== Sargan (overidentifisering) ====")
print(tsls_model.sargan)
print("\n==== Hovedkoeffisient ====")
print(f"beta_{IV_ENDOG} = {tsls_model.params[IV_ENDOG]:.4f}")
print(f"std err       = {tsls_model.std_errors[IV_ENDOG]:.4f}")
print(f"t-statistikk  = {tsls_model.tstats[IV_ENDOG]:.3f}")
print(f"p-verdi       = {tsls_model.pvalues[IV_ENDOG]:.3g}")

==== Førstesteg ====
    First Stage Estimation Results    
                              cons_NO4
--------------------------------------
R-squared                       0.8344
Partial R-squared               0.2482
Shea's R-squared                0.2482
Partial F-statistic             428.06
P-value (Partial F-stat)        0.0000
Partial F-stat Distn           chi2(2)
========================== ===========
const                           2518.8
                              (168.00)
fill_avvik                     -872.59
                             (-12.387)
prod_wind_NO4                   0.1001
                              (4.5828)
D_m2                            29.361
                              (1.6204)
D_m3                           -56.700
                             (-2.8624)
D_m4                           -209.55
                             (-10.944)
D_m5                           -375.04
                             (-16.862)
D_m6                           -424.24
    

In [5]:
y_test = df_iso.loc[test_mask, TARGET]
preds_test = predict_2sls(
    result=tsls_model,
    df=df_iso.loc[test_mask],
    endog_col=IV_ENDOG,
    exog_cols=IV_EXOG,
)

preds = make_prediction_frame(
    df=df_iso,
    mask=test_mask,
    actual=y_test,
    prediction_col="TSLS",
    predictions=preds_test.reindex(df_iso.loc[test_mask].index),
)
metrics = eval_metrics(preds["actual"], preds["TSLS"])
display(pd.DataFrame([metrics], index=["TSLS"]))

,MAE,RMSE,R²,N
TSLS,185.8,249.4,0.0508,9844


In [6]:
payload = {
    "model_name": "TSLS",
    "prediction_col": "TSLS",
    "endog_col": IV_ENDOG,
    "instrument_cols": IV_INSTRUMENTS,
    "exog_cols": IV_EXOG,
    "target_col": TARGET,
    "train_years": TRAIN_YEARS,
    "test_years": TEST_YEARS,
    "estimator": tsls_model,
    "metrics": metrics,
    "diagnostics": {
        "beta_endog": float(tsls_model.params[IV_ENDOG]),
        "se_endog": float(tsls_model.std_errors[IV_ENDOG]),
        "t_endog": float(tsls_model.tstats[IV_ENDOG]),
        "p_endog": float(tsls_model.pvalues[IV_ENDOG]),
        "sargan_stat": float(tsls_model.sargan.stat),
        "sargan_pval": float(tsls_model.sargan.pval),
        "first_stage_partial_F": float(tsls_model.first_stage.diagnostics.loc[IV_ENDOG, "f.stat"]) if hasattr(tsls_model.first_stage, "diagnostics") else None,
    },
}

save_model_artifacts("tsls", preds, payload, intermediate_dir=INTERMEDIATE_DIR)
print(f"Lagret {INTERMEDIATE_DIR}preds_tsls.parquet")
print(f"Lagret {INTERMEDIATE_DIR}models_tsls.pkl")

Lagret intermediate/preds_tsls.parquet
Lagret intermediate/models_tsls.pkl
